# Homework 1: Agentic RAG

## Setup

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [ ]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

## Q1. How many lesson pages?

How many lesson pages are in the dataset?
* 24
* **72**
* 240
* 720

In [ ]:
len(documents)

## Q2. Indexing and searching

What's the filename of the first result?

* 01-agentic-rag/lessons/03-rag.md
* **01-agentic-rag/lessons/14-agentic-loop.md**
* 04-evaluation/lessons/13-llm-as-judge.md
* 06-best-practices/lessons/02-hybrid-search.md

In [ ]:
from minsearch import Index

def build_index(documents):
    index = Index(
        text_fields=['content'],
        keyword_fields=['filename']
    )
    index.fit(documents)
    return index

index = build_index(documents)
index

In [ ]:
query = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(query)
[doc['filename'] for doc in search_results][0]

## Q3. RAG

Using gpt-5.4-mini, how many input (prompt) tokens did we send to the model for this request?

* 700
* **7000**
* 70000
* 700000

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from rag_helper_modified import RAGBase

In [ ]:
load_dotenv()
openai_client = OpenAI()

In [ ]:
assistant = RAGBase(index, openai_client)
query = "How does the agentic loop keep calling the model until it stops?"
answer = assistant.rag(query)
print(f"Prompt tokens: {answer.input_tokens}")

## Q4. Chunking

How many chunks do you get?
* 70
* **295**
* 1100
* 4500

In [ ]:
from gitsource import chunk_documents

In [ ]:
chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

## Q5. RAG with chunking

Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?
* about the same
* **3x fewer**
* 10x fewer
* 30x fewer

In [ ]:
chunked_index = build_index(chunks)
chunked_index

In [ ]:
chunked_assistant = RAGBase(chunked_index, openai_client)
chunked_answer = chunked_assistant.rag(query)
print(f"Prompt tokens: {chunked_answer.input_tokens}")

## Q6. Turning it into an agent

How many times did the agent call search?
* 0
* 4
* **10**
* 20

In [ ]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [ ]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

instructions = """
You're a course teaching assistant.
    
Answer the student's question using the search tool.
    
Make multiple searches with different keywords before answering.
""".strip()

In [ ]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)
agent_tools.get_tools()

In [ ]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)